# Train the steerdb Tree-CNN on a free GPU (Colab / Kaggle)

This notebook only **installs and calls the `steerdb` package**; all logic lives in the repo.
Training needs just the experience store (`runs/experience.sqlite`), not Postgres.

Workflow:
1. Locally: `steerdb collect` → produces `runs/experience.sqlite`.
2. Here: install the repo, upload the store, run `steerdb train` / `steerdb bench` on the GPU.
3. Download the trained model and use it locally (`steerdb run`, `steerdb bench`).
   Weights are saved as CPU tensors, so they load on a machine without a GPU.

Colab: *Runtime → Change runtime type → GPU*. Kaggle: *Settings → Accelerator → GPU*.

## 1. Get the code
Either clone from GitHub, or (before the repo is public) upload a zip made locally with
`git archive -o steerdb.zip HEAD`.

In [ ]:
import os
import pathlib
import subprocess
import zipfile

REPO_URL = ""  # e.g. "https://github.com/<you>/steerdb.git"; leave empty to use an uploaded zip
ROOT = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle") else "/content")
REPO = ROOT / "steerdb"

if not REPO.exists():
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    else:
        zips = [
            p
            for p in [ROOT / "steerdb.zip", *pathlib.Path("/kaggle/input").glob("**/steerdb.zip")]
            if p.exists()
        ]
        if not zips:
            from google.colab import files  # Colab only; on Kaggle attach the zip as a dataset

            files.upload()
            zips = [ROOT / "steerdb.zip"]
        zipfile.ZipFile(zips[0]).extractall(REPO)
os.chdir(REPO)
print("repo at", REPO)

## 2. Install the package (torch is preinstalled on Colab/Kaggle) and fetch the JOB query names

In [ ]:
%pip install -q -e .
!bash workload/fetch_job.sh

In [ ]:
import torch

print(
    "CUDA available:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

## 3. Provide the experience store
Upload `runs/experience.sqlite` from your machine (Colab), or attach it as a Kaggle dataset.

In [ ]:
STORE = REPO / "runs" / "experience.sqlite"
STORE.parent.mkdir(parents=True, exist_ok=True)
if not STORE.exists():
    found = (
        list(pathlib.Path("/kaggle/input").glob("**/experience.sqlite"))
        if os.path.exists("/kaggle/input")
        else []
    )
    if found:
        STORE.write_bytes(found[0].read_bytes())
    else:
        from google.colab import files

        up = files.upload()  # pick runs/experience.sqlite
        STORE.write_bytes(next(iter(up.values())))
os.environ["STEERDB_STORE"] = str(STORE)
!steerdb oracle-gap

## 4. Train on the GPU

In [ ]:
!steerdb train --model lgbm
!steerdb train --model treecnn --device cuda

## 5. (Optional) Full offline evaluation + ablations on the GPU
Every Tree-CNN in the evaluation (main, ablations, online replay) trains on the GPU automatically.
`--overhead` is omitted because it needs a live Postgres.

In [ ]:
!steerdb bench --out runs/eval --markdown runs/eval/results.md

## 6. Download the results

In [ ]:
import shutil

archive = shutil.make_archive(str(ROOT / "steerdb_outputs"), "zip", REPO / "runs", ".")
print(archive)
try:
    from google.colab import files

    files.download(archive)
except ImportError:
    print("Kaggle: download it from the Output tab")

Locally: unzip into `runs/` (you get `runs/models/treecnn`, `runs/models/lgbm`, `runs/eval/`), then e.g.
`steerdb run --file workload/job/29a.sql`. To publish evaluation results, copy `runs/eval/results.json`
to `docs/` and run `python experiments/plots.py`, or rerun `python experiments/run_eval.py --overhead`
locally.